# Week 8 — Evaluation Expansion

This notebook evaluates the final CatBoost model using additional
percentage-based metrics and examines performance across property-price
bands.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

y_test = np.load("week7_y_test.npy")
catboost_predictions = np.load("week7_catboost_predictions.npy")

print("Actual prices:", y_test.shape)
print("Predictions:", catboost_predictions.shape)

assert len(y_test) == len(catboost_predictions)

Absolute percentage error =
|actual price - predicted price| / actual price × 100

MAPE - the mean percentage error across all properties 

MdAPE - the median percentage error

In [ ]:
absolute_percentage_errors = (
    np.abs(y_test - catboost_predictions)
    / y_test
) * 100

print(
    "First 10 percentage errors:",
    absolute_percentage_errors[:10]
)

In [ ]:
r2 = r2_score(y_test, catboost_predictions)

mae = mean_absolute_error(
    y_test,
    catboost_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        catboost_predictions
    )
)

mape = absolute_percentage_errors.mean()

mdape = np.median(
    absolute_percentage_errors
)

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape, "%")
print("MdAPE:", mdape, "%")

In [ ]:
evaluation_df = pd.DataFrame({
    "ActualPrice": y_test,
    "PredictedPrice": catboost_predictions
})

evaluation_df["AbsoluteError"] = np.abs(
    evaluation_df["ActualPrice"]
    - evaluation_df["PredictedPrice"]
)

evaluation_df["AbsolutePercentageError"] = (
    evaluation_df["AbsoluteError"]
    / evaluation_df["ActualPrice"]
) * 100

evaluation_df.head()

In [ ]:
price_edges = [
    0,
    500_000,
    750_000,
    1_000_000,
    1_500_000,
    2_000_000,
    np.inf
]

price_labels = [
    "Under $500K",
    "$500K-$750K",
    "$750K-$1M",
    "$1M-$1.5M",
    "$1.5M-$2M",
    "$2M+"
]

evaluation_df["PriceBand"] = pd.cut( # reads one ActualPrice at a time and add to each boundary
    evaluation_df["ActualPrice"],
    bins=price_edges,
    labels=price_labels,
    include_lowest=True,
    right=False
)

### **Calculate Overall Metrics**

In [ ]:
def calculate_regression_metrics(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)

    absolute_percentage_errors = (
        np.abs(actual- predicted) / actual
    ) * 100

    return {
        "R2": r2_score(actual, predicted),
        "MAE" : mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE": absolute_percentage_errors.mean(),
        "MdAPE": np.median(absolute_percentage_errors)
    }

In [ ]:
band_results = []

for price_band, group in evaluation_df.groupby(
    "PriceBand",
    observed=True # Include only price bands that actually contain propertis
):
    metrics = calculate_regression_metrics(
        group["ActualPrice"],
        group["PredictedPrice"]
    )
    band_results.append({
        "Model": "CatBoost",
        "Segment": str(price_band),
        "NumberOfProperties" : len(group),
        **metrics
    })

price_band_metrics = pd.DataFrame(
    band_results
)

price_band_metrics

In [ ]:
overall_metrics = calculate_regression_metrics(
    y_test,
    catboost_predictions
)

overall_row = pd.DataFrame([{
    "Model": "CatBoost",
    "Segment": "Overall",
    "NumberOfProperties": len(y_test),
    **overall_metrics
}])
overall_row

In [ ]:
metrics_summary = pd.concat(
    [overall_row, price_band_metrics],
    ignore_index=True
)

metrics_summary

In [ ]:
metrics_summary.to_csv(
    "catboost_metrics_summary.csv",
    index=False
)